<a href="https://colab.research.google.com/github/vkjadon/hugging_face/blob/main/hf_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Enable GPU Runtime

Navigate to Runtime → Change runtime type → T4 GPU → Save.



Verify GPU access:

In [1]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


## Install Libraries

The -q flag suppresses verbose output.

In [2]:
!pip install -q transformers datasets huggingface_hub accelerate

## Authentication
For private models or pushing to Hub, authenticate with your token:

In [ ]:
from huggingface_hub import login
login()  # Opens interactive prompt

For a cleaner workflow, use Colab secrets:

To set up secrets, click the key icon in Colab's left sidebar and add HF_TOKEN with your Hugging Face access token.

In [3]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

In [13]:
from huggingface_hub import list_datasets

# Get all available datasets
all_datasets = list_datasets()

# Search for specific datasets by name
imdb_datasets = list_datasets(search="imdb")
print(imdb_datasets)
print(dir(imdb_datasets))

<generator object HfApi.list_datasets at 0x7c9383a618c0>
['__class__', '__del__', '__delattr__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__lt__', '__name__', '__ne__', '__new__', '__next__', '__qualname__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', 'close', 'gi_code', 'gi_frame', 'gi_running', 'gi_suspended', 'gi_yieldfrom', 'send', 'throw']


In [ ]:
for dataset in imdb_datasets:
    print(dataset.id)

In [ ]:
import pprint

# Get the first item from the generator
first_item = next(list_datasets(search="imdb"))

# Print all available fields and methods
print(dir(first_item))


# Convert the object's attributes to a readable dictionary
pprint.pprint(vars(first_item))


Download full dataset

In [ ]:
from datasets import load_dataset

In [27]:
help(load_dataset)

Help on function load_dataset in module datasets.load:

load_dataset(path: str, name: Optional[str] = None, data_dir: Optional[str] = None, data_files: Union[str, collections.abc.Sequence[str], collections.abc.Mapping[str, Union[str, collections.abc.Sequence[str]]], NoneType] = None, split: Union[str, datasets.splits.Split, list[str], list[datasets.splits.Split], NoneType] = None, cache_dir: Optional[str] = None, features: Optional[datasets.features.features.Features] = None, download_config: Optional[datasets.download.download_config.DownloadConfig] = None, download_mode: Union[datasets.download.download_manager.DownloadMode, str, NoneType] = None, verification_mode: Union[datasets.utils.info_utils.VerificationMode, str, NoneType] = None, keep_in_memory: Optional[bool] = None, save_infos: bool = False, revision: Union[str, datasets.utils.version.Version, NoneType] = None, token: Union[bool, str, NoneType] = None, streaming: bool = False, num_proc: Optional[int] = None, storage_options

The load_dataset function loads datasets from the Hugging Face Hub or local files, with path serving as the mandatory identifier for the dataset repository or file format. Key arguments include split for requesting specific data portions, data_files for mapping local files, and streaming to enable on-the-fly data loading without downloading the full dataset

In [30]:
dataset = load_dataset("stanfordnlp/imdb")

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


This is a DatasetDict object, which acts like a Python dictionary organizing your dataset into three separate parts (splits):

* train: 25,000 labeled rows used to train your model.
* test: 25,000 labeled rows used to evaluate your model's performance.
* unsupervised: 50,000 unlabeled rows meant for extra pre-training or self-supervised learning.
* features: Every single row across all splits contains exactly two columns: text (the movie review string) and label (the sentiment score).

Would you like to see how to access a specific row of text from the training split, or see how the labels are mapped to positive and negative?



Download a subset

In [36]:
dataset = load_dataset("stanfordnlp/imdb", split={"train" : "train[:1000]", "test" : "test[:1000]"})
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1000
    })
})


In [37]:
dataset = load_dataset("stanfordnlp/imdb", split=["train[:1000]", "test[:1000]"])
print(dataset)

[Dataset({
    features: ['text', 'label'],
    num_rows: 1000
}), Dataset({
    features: ['text', 'label'],
    num_rows: 1000
})]


Pass a list of strings to the split argument. This returns a standard Python list containing the individual dataset splits in the same order.

In [18]:
train_data = dataset["train"]
example = train_data[0]
print(example)

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [ ]:
for example in train_data["label"]:
    print(example)

In [ ]:
texts = train_data["text"]

print(texts[:5])

Load a Model Pipeline

In [ ]:
from transformers import pipeline

classifier = pipeline("text-classification", device=0)  # device=0 uses GPU
result = classifier("I love this course!")
print(result)

Load a Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("imdb", split="train[:100]")
print(dataset[0])

Check Device Placement


In [ ]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0)}")

## Memory Management
Colab's free tier provides approximately 15GB of GPU memory on the T4. Use these techniques to work within that limit.

In [ ]:
import torch
import gc

del model  # Delete the model variable
gc.collect()
torch.cuda.empty_cache()

##Load in Lower Precision

In [ ]:
from transformers import AutoModelForCausalLM
import torch

model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-2-7b-hf",
    torch_dtype=torch.float16,
    device_map="auto"
)

##Mount Google Drive
Save models and checkpoints to persist across sessions:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Save model
model.save_pretrained('/content/drive/MyDrive/my_model')

Download Files Locally

In [ ]:
from google.colab import files
files.download('output.csv')

In [ ]:
# 2. Search for models
from huggingface_hub import list_models

models = list(list_models(pipeline_tag="text-classification", sort="downloads", limit=5))
for m in models:
    print(f"{m.id}: {m.downloads:,} downloads")

# 3. Try the top model
from transformers import pipeline

In [ ]:
classifier = pipeline("text-classification", model=models[0].id, device=0)
print(classifier("This is amazing!"))

In [ ]:
classifier = pipeline("text-classification", model=models[2].id, device=0)
print(classifier("This is amazing!"))

## Streaming

In [ ]:
from datasets import load_dataset

In [ ]:
def load_and_stream_filter(dataset_name: str, split_name: str, keyword: str):
    """Stream a dataset split and collect up to 5 'text' values containing the keyword.

    Args:
        dataset_name: The name or path of the dataset on the Hub.
        split_name: The name of the split to stream (e.g., 'train').
        keyword: Substring to search for in the 'text' field.

    Returns:
        A list of up to 5 strings from the 'text' column that contain the keyword.
    """
    # 1. Enable streaming to avoid downloading the entire dataset
    dataset = load_dataset(dataset_name, split=split_name, streaming=True)

    results = []

    # 2. Iterate over the stream
    for item in dataset:
        text_content = item.get("text", "")

        # 3. Check for the keyword substring
        if keyword in text_content:
            results.append(text_content)

            # Break early once we hit the 5-string limit
            if len(results) == 5:
                break

    # 4. Return the filtered list
    return results


## Search for positive sentiment markers in movie reviews

In [ ]:
results = load_and_stream_filter(dataset_name="rotten_tomatoes", split_name="train", keyword="masterpiece")

In [ ]:
print(f"Found {len(results)} matches:")
for i, text in enumerate(results, 1):
    print(f"{i}. {text}")

## Optimized Implementation with .filter() and .take()

Hugging Face IterableDataset objects have a built-in .take(n) method. It automatically limits the stream to the first n elements. However, because you need to filter by a keyword before counting to 5, you must combine it with .filter(). If you use .take(5) first, you will only look at the first 5 rows of the dataset, and if they don't contain the keyword, your function will return nothing.

Here is how you can rewrite the function cleanly using the library's core streaming operations:

In [ ]:
def load_and_stream_filter(dataset_name: str, split_name: str, keyword: str):
    # 1. Load the streaming dataset
    dataset = load_dataset(dataset_name, split=split_name, streaming=True)

    # 2. Apply a streaming filter (lazy evaluation)
    filtered_stream = dataset.filter(lambda item: keyword in item.get("text", ""))

    # 3. Limit the stream to the first 5 matches and extract the 'text' field
    # We use a list comprehension to pull the items out of the stream
    results = [item["text"] for item in filtered_stream.take(5)]

    return results

In [ ]:
results = load_and_stream_filter(dataset_name="rotten_tomatoes", split_name="train", keyword="masterpiece")

In [38]:
import gradio as gr
from transformers import pipeline

classifier = pipeline("sentiment-analysis")

def predict(text):
    return classifier(text)

demo = gr.Interface(
    fn=predict,
    inputs="text",
    outputs="json"
)

demo.launch()

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b5cb14e4c9799d0d93.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [39]:
import gradio as gr

# Define the function your UI will run
def analyze_review(text):
    word_count = len(text.split())
    sentiment = "Positive Tone 😃" if "good" in text.lower() or "great" in text.lower() else "Neutral/Negative Tone 😐"
    return f"Word Count: {word_count}", f"Predicted Sentiment: {sentiment}"

# Build the UI layout
demo = gr.Interface(
    fn=analyze_review,
    inputs=gr.Textbox(lines=3, placeholder="Enter a movie review here..."),
    outputs=["text", "text"],
    title="IMDB Review Analyzer"
)

# Launch the app inside Colab
demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8bea5ad7529068b0c2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
